In [ ]:
# ==============================
# 🧠 Deep Learning - Video Game Sales Prediction
# ==============================

import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from preprocessing import load_data, clean_data, encode_data, split_features

# ==============================
# CARGA DE DATOS
# ==============================

df = load_data("../data/ventas_videojuegos.csv")
df = clean_data(df)
df = encode_data(df)

print("Shape:", df.shape)

# ==============================
# FEATURES Y TARGET
# ==============================

X, y = split_features(df)

# ❗ ELIMINAR DATA LEAKAGE
X = X.drop(["Ventas NA", "Ventas EU", "Ventas JP", "Ventas Otros"], axis=1)

# ==============================
# SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# NORMALIZACIÓN
# ==============================

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==============================
# MODELO
# ==============================

model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print(model.summary())

# ==============================
# ENTRENAMIENTO
# ==============================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("Entrenando modelo...")

history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

print("Modelo entrenado!")

# ==============================
# VISUALIZACIÓN
# ==============================

plt.figure()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.legend()
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")

plt.show()

# ==============================
# INTERPRETACIÓN
# ==============================

print("\n📊 Model Evaluation:")
print("- Training loss decreases significantly")
print("- Validation loss remains high")

print("\n🧠 Insight:")
print("The model shows overfitting. It learns training data well but fails to generalize.")
print("This suggests the limitation is in the dataset rather than the model.")

ModuleNotFoundError: No module named 'preprocessing'